# Genetic Algorithm Optimization for CNN-LSTM Battery Health Prediction

This notebook implements a Genetic Algorithm to optimize hyperparameters for the Sequential CNN-LSTM architecture.

## What is Genetic Algorithm?

Genetic Algorithms (GA) are inspired by natural evolution:
- **Population**: Set of candidate solutions (model configurations)
- **Genes**: Hyperparameters (CNN layers, LSTM units, learning rate, etc.)
- **Fitness**: Model performance (validation RMSE)
- **Selection**: Best performers are chosen to reproduce
- **Crossover**: Combine genes from two parents
- **Mutation**: Random changes to explore new configurations
- **Evolution**: Iterate over generations to find optimal solution

## GA vs Bayesian Optimization

| Aspect | Bayesian Optimization | Genetic Algorithm |
|--------|----------------------|-------------------|
| **Approach** | Probabilistic model | Evolutionary |
| **Exploration** | Guided by surrogate model | Random + crossover |
| **Parallelization** | Sequential | Naturally parallel |
| **Best for** | Continuous spaces | Mixed spaces |
| **Interpretability** | Lower | Higher (see evolution) |

---

## Configuration

**Adjust these parameters before running:**

In [1]:
# GA Configuration
POPULATION_SIZE = 20      # Number of individuals per generation
GENERATIONS = 15          # Number of evolution cycles
CROSSOVER_RATE = 0.7      # Probability of combining two parents
MUTATION_RATE = 0.2       # Probability of random gene change
ELITE_SIZE = 2            # Top performers carried to next generation
TOURNAMENT_SIZE = 3       # Selection tournament size
MAX_EPOCHS_PER_MODEL = 100  # Maximum training epochs per individual

# Estimated runtime
estimated_models = POPULATION_SIZE * GENERATIONS
estimated_time_hours = (estimated_models * 30) / 3600  # 30 seconds per model

print(f"Configuration:")
print(f"  Total models to train: {estimated_models}")
print(f"  Estimated time: {estimated_time_hours:.1f} hours")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Generations: {GENERATIONS}")

Configuration:
  Total models to train: 300
  Estimated time: 2.5 hours
  Population size: 20
  Generations: 15


## 1. Setup and Imports

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import datetime
import pickle
import json
import random
from copy import deepcopy
import time as time_module  # Renamed to avoid conflict with 'time' variable

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# Create output directories
os.makedirs('output_ga', exist_ok=True)
os.makedirs('models_ga', exist_ok=True)
os.makedirs('ga_generations', exist_ok=True)

print("✓ All imports successful")
print(f"✓ TensorFlow version: {tf.__version__}")
print(f"✓ GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

✓ All imports successful
✓ TensorFlow version: 2.10.0
✓ GPU available: True


## 2. Load and Prepare Data

Load actual NASA B0005 battery dataset and apply improved data split

In [3]:
# Load NASA B0005 battery aging dataset (.mat file)
# Dataset path - adjust if your data is in a different location
import scipy.io

DATA_PATH = 'B0005.mat'

print(f"Loading NASA B0005 dataset from: {DATA_PATH}")

# Load .mat file
mat_data = scipy.io.loadmat(DATA_PATH)
B0005 = mat_data['B0005']
cycles = B0005['cycle'][0, 0]

# Extract all discharge cycles
discharge_data = []
discharge_cycle_num = 0

for i in range(cycles.shape[1]):
    cycle = cycles[0, i]
    cycle_type = str(cycle['type'][0])
    
    if 'discharge' in cycle_type.lower():
        discharge_cycle_num += 1
        ambient_temp = float(cycle['ambient_temperature'][0])
        data = cycle['data'][0]
        
        # Extract measurements
        voltage = data['Voltage_measured'][0].flatten()
        current = data['Current_measured'][0].flatten()
        temperature = data['Temperature_measured'][0].flatten()
        current_load = data['Current_load'][0].flatten()
        voltage_load = data['Voltage_load'][0].flatten()
        time = data['Time'][0].flatten()
        capacity = float(data['Capacity'][0][0, 0])
        
        # Create records for each measurement
        for j in range(len(voltage)):
            discharge_data.append({
                'cycle': discharge_cycle_num,
                'capacity': capacity,
                'voltage_measured': voltage[j],
                'current_measured': current[j],
                'temperature_measured': temperature[j],
                'current_load': current_load[j],
                'voltage_load': voltage_load[j],
                'time': time[j],
                'ambient_temperature': ambient_temp
            })

# Create DataFrame
discharge_df = pd.DataFrame(discharge_data)

# Calculate SoH
C_initial = discharge_df.groupby('cycle')['capacity'].first().iloc[0]
discharge_df['SoH'] = discharge_df['capacity'] / C_initial

# Get statistics
n_cycles = discharge_df['cycle'].nunique()

print(f"✓ Data loaded: {len(discharge_df):,} measurements, {n_cycles} cycles")
print(f"  - Initial capacity: {C_initial:.4f} Ah")
print(f"  - Final capacity: {discharge_df.groupby('cycle')['capacity'].first().iloc[-1]:.4f} Ah")

Loading NASA B0005 dataset from: B0005.mat
✓ Data loaded: 50,285 measurements, 168 cycles
  - Initial capacity: 1.8565 Ah
  - Final capacity: 1.3251 Ah


In [4]:
# Create sequences
features = ['voltage_measured', 'current_measured', 'temperature_measured', 
            'current_load', 'voltage_load', 'time']

def create_sequences(df, sequence_length=50):
    sequences = []
    targets = []
    cycles = []
    
    for cycle in df['cycle'].unique():
        cycle_data = df[df['cycle'] == cycle][features].values
        soh = df[df['cycle'] == cycle]['SoH'].iloc[0]
        
        if len(cycle_data) >= sequence_length:
            indices = np.linspace(0, len(cycle_data)-1, sequence_length, dtype=int)
            sequence = cycle_data[indices]
            sequences.append(sequence)
            targets.append(soh)
            cycles.append(cycle)
    
    return np.array(sequences), np.array(targets), np.array(cycles)

SEQUENCE_LENGTH = 50
X, y, cycle_ids = create_sequences(discharge_df, SEQUENCE_LENGTH)

# Improved data split
train_cycles = list(range(1, 111)) + list(range(145, 156))
val_cycles = list(range(111, 131))
test_cycles = list(range(131, 145)) + list(range(156, 169))

train_mask = np.isin(cycle_ids, train_cycles)
val_mask = np.isin(cycle_ids, val_cycles)
test_mask = np.isin(cycle_ids, test_cycles)

X_train = X[train_mask]
y_train = y[train_mask]
X_val = X[val_mask]
y_val = y[val_mask]
X_test = X[test_mask]
y_test = y[test_mask]

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

print(f"✓ Data prepared: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

✓ Data prepared: Train=121, Val=20, Test=27


## 3. Define Gene Space

These are the hyperparameters that will be optimized

In [5]:
GENE_SPACE = {
    'n_cnn_layers': [1, 2, 3, 4, 5],
    'cnn_filters': [32, 64, 128, 256, 512],
    'kernel_size': [3, 5, 7, 9, 11],
    'cnn_activation': ['relu', 'elu', 'selu', 'tanh'],
    'n_lstm_layers': [1, 2, 3, 4, 5],
    'lstm_units': [32, 64, 128, 256],
    'lstm_activation': ['relu', 'tanh', 'sigmoid'],
    'dropout_rate': (0.1, 0.5),  # continuous range
    'dense_units': [32, 64, 128, 256],
    'learning_rate': (0.0001, 0.01),  # log scale
    'batch_size': [16, 32, 64]
}

print("Gene Space Defined:")
for gene, space in GENE_SPACE.items():
    if isinstance(space, tuple):
        print(f"  {gene}: {space[0]} - {space[1]} (continuous)")
    else:
        print(f"  {gene}: {space}")

Gene Space Defined:
  n_cnn_layers: [1, 2, 3, 4, 5]
  cnn_filters: [32, 64, 128, 256, 512]
  kernel_size: [3, 5, 7, 9, 11]
  cnn_activation: ['relu', 'elu', 'selu', 'tanh']
  n_lstm_layers: [1, 2, 3, 4, 5]
  lstm_units: [32, 64, 128, 256]
  lstm_activation: ['relu', 'tanh', 'sigmoid']
  dropout_rate: 0.1 - 0.5 (continuous)
  dense_units: [32, 64, 128, 256]
  learning_rate: 0.0001 - 0.01 (continuous)
  batch_size: [16, 32, 64]


## 4. Define Individual Class

In [6]:
class Individual:
    """Represents one individual (model configuration) in the population"""
    
    def __init__(self, genes=None):
        if genes is None:
            self.genes = self.random_genes()
        else:
            self.genes = genes
        self.fitness = None
        self.model = None
        self.training_time = 0
        
    def random_genes(self):
        """Generate random genes"""
        genes = {}
        
        genes['n_cnn_layers'] = random.choice(GENE_SPACE['n_cnn_layers'])
        genes['cnn_filters'] = [random.choice(GENE_SPACE['cnn_filters']) for _ in range(5)]
        genes['kernel_size'] = random.choice(GENE_SPACE['kernel_size'])
        genes['cnn_activation'] = random.choice(GENE_SPACE['cnn_activation'])
        
        genes['n_lstm_layers'] = random.choice(GENE_SPACE['n_lstm_layers'])
        genes['lstm_units'] = [random.choice(GENE_SPACE['lstm_units']) for _ in range(5)]
        genes['lstm_activation'] = random.choice(GENE_SPACE['lstm_activation'])
        
        genes['dropout_rate'] = random.uniform(*GENE_SPACE['dropout_rate'])
        genes['dense_units'] = random.choice(GENE_SPACE['dense_units'])
        genes['learning_rate'] = 10 ** random.uniform(
            np.log10(GENE_SPACE['learning_rate'][0]), 
            np.log10(GENE_SPACE['learning_rate'][1])
        )
        genes['batch_size'] = random.choice(GENE_SPACE['batch_size'])
        
        return genes
    
    def build_model(self, input_shape):
        """Build Sequential CNN-LSTM model from genes"""
        inputs = layers.Input(shape=input_shape)
        x = inputs
        
        # CNN layers
        for i in range(self.genes['n_cnn_layers']):
            filters = self.genes['cnn_filters'][i]
            x = layers.Conv1D(filters, self.genes['kernel_size'], 
                             activation=self.genes['cnn_activation'], padding='same')(x)
            x = layers.BatchNormalization()(x)
            x = layers.MaxPooling1D(2)(x)
            x = layers.Dropout(self.genes['dropout_rate'])(x)
        
        # LSTM layers
        for i in range(self.genes['n_lstm_layers']):
            units = self.genes['lstm_units'][i]
            return_sequences = (i < self.genes['n_lstm_layers'] - 1)
            x = layers.LSTM(units, activation=self.genes['lstm_activation'], 
                           return_sequences=return_sequences)(x)
            x = layers.Dropout(self.genes['dropout_rate'])(x)
        
        # Dense layers
        x = layers.Dense(self.genes['dense_units'], activation='relu')(x)
        x = layers.Dropout(self.genes['dropout_rate'])(x)
        outputs = layers.Dense(1, activation='linear')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        optimizer = keras.optimizers.Adam(learning_rate=self.genes['learning_rate'])
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
        
        return model
    
    def evaluate(self, X_train, y_train, X_val, y_val, max_epochs=MAX_EPOCHS_PER_MODEL):
        """Train and evaluate the model"""
        try:
            start_time = time_module.time()
            
            self.model = self.build_model(input_shape=(X_train.shape[1], X_train.shape[2]))
            
            early_stop = EarlyStopping(monitor='val_loss', patience=15, 
                                      restore_best_weights=True, verbose=0)
            reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                                         patience=8, min_lr=1e-6, verbose=0)
            
            history = self.model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=max_epochs,
                batch_size=self.genes['batch_size'],
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )
            
            y_pred = self.model.predict(X_val, verbose=0).flatten()
            val_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            
            self.fitness = -val_rmse
            self.training_time = time_module.time() - start_time
            
            return val_rmse
            
        except Exception as e:
            print(f"  ⚠️  Model training failed: {str(e)}")
            self.fitness = -999
            return 999
    
    def __repr__(self):
        return f"Individual(fitness={self.fitness:.6f if self.fitness else 'None'})"

print("✓ Individual class defined")

✓ Individual class defined


## 5. Define GA Operators

In [7]:
def tournament_selection(population, tournament_size=TOURNAMENT_SIZE):
    """Select individual using tournament selection"""
    tournament = random.sample(population, tournament_size)
    return max(tournament, key=lambda ind: ind.fitness if ind.fitness else -999)


def crossover(parent1, parent2):
    """Perform crossover between two parents"""
    if random.random() > CROSSOVER_RATE:
        return deepcopy(parent1), deepcopy(parent2)
    
    child1_genes = {}
    child2_genes = {}
    
    for key in parent1.genes:
        if random.random() < 0.5:
            child1_genes[key] = deepcopy(parent1.genes[key])
            child2_genes[key] = deepcopy(parent2.genes[key])
        else:
            child1_genes[key] = deepcopy(parent2.genes[key])
            child2_genes[key] = deepcopy(parent1.genes[key])
    
    return Individual(child1_genes), Individual(child2_genes)


def mutate(individual):
    """Mutate an individual's genes"""
    if random.random() > MUTATION_RATE:
        return individual
    
    gene_keys = list(individual.genes.keys())
    gene_to_mutate = random.choice(gene_keys)
    
    if gene_to_mutate == 'n_cnn_layers':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['n_cnn_layers'])
    elif gene_to_mutate == 'cnn_filters':
        idx = random.randint(0, 4)
        individual.genes[gene_to_mutate][idx] = random.choice(GENE_SPACE['cnn_filters'])
    elif gene_to_mutate == 'kernel_size':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['kernel_size'])
    elif gene_to_mutate == 'cnn_activation':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['cnn_activation'])
    elif gene_to_mutate == 'n_lstm_layers':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['n_lstm_layers'])
    elif gene_to_mutate == 'lstm_units':
        idx = random.randint(0, 4)
        individual.genes[gene_to_mutate][idx] = random.choice(GENE_SPACE['lstm_units'])
    elif gene_to_mutate == 'lstm_activation':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['lstm_activation'])
    elif gene_to_mutate == 'dropout_rate':
        individual.genes[gene_to_mutate] = random.uniform(*GENE_SPACE['dropout_rate'])
    elif gene_to_mutate == 'dense_units':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['dense_units'])
    elif gene_to_mutate == 'learning_rate':
        individual.genes[gene_to_mutate] = 10 ** random.uniform(
            np.log10(GENE_SPACE['learning_rate'][0]), 
            np.log10(GENE_SPACE['learning_rate'][1])
        )
    elif gene_to_mutate == 'batch_size':
        individual.genes[gene_to_mutate] = random.choice(GENE_SPACE['batch_size'])
    
    return individual

print("✓ GA operators defined (selection, crossover, mutation)")

✓ GA operators defined (selection, crossover, mutation)


## 6. Run Genetic Algorithm

**This will take several hours!** Monitor progress below.

In [8]:
# Initialize tracking
evolution_history = {
    'generations': [],
    'best_fitness': [],
    'avg_fitness': [],
    'best_genes': [],
    'best_rmse': []
}

# Initialize population
print(f"Initializing population of {POPULATION_SIZE} individuals...")
population = [Individual() for _ in range(POPULATION_SIZE)]
print("✓ Population initialized\n")

# Evolution loop
for generation in range(GENERATIONS):
    gen_start_time = time_module.time()
    
    print(f"{'='*80}")
    print(f"GENERATION {generation + 1}/{GENERATIONS}")
    print(f"{'='*80}")
    
    # Evaluate population
    fitness_scores = []
    rmse_scores = []
    
    for i, individual in enumerate(population):
        if individual.fitness is None:
            print(f"  [{i+1}/{POPULATION_SIZE}] Evaluating individual...")
            rmse = individual.evaluate(X_train_scaled, y_train, X_val_scaled, y_val)
            fitness_scores.append(individual.fitness)
            rmse_scores.append(rmse)
            print(f"      Validation RMSE: {rmse:.6f}")
        else:
            fitness_scores.append(individual.fitness)
            rmse_scores.append(-individual.fitness)
    
    # Sort population
    population.sort(key=lambda ind: ind.fitness, reverse=True)
    
    # Track best
    best_individual = population[0]
    best_rmse = -best_individual.fitness
    avg_fitness = np.mean(fitness_scores)
    
    evolution_history['generations'].append(generation + 1)
    evolution_history['best_fitness'].append(best_individual.fitness)
    evolution_history['avg_fitness'].append(avg_fitness)
    evolution_history['best_genes'].append(deepcopy(best_individual.genes))
    evolution_history['best_rmse'].append(best_rmse)
    
    # Save best model
    best_individual.model.save(f'ga_generations/best_gen_{generation+1}.keras')
    
    gen_time = time_module.time() - gen_start_time
    
    print(f"\n  Generation {generation + 1} Summary:")
    print(f"    Best RMSE:    {best_rmse:.6f}")
    print(f"    Avg Fitness:  {avg_fitness:.6f}")
    print(f"    Time:         {gen_time:.1f}s")
    print()
    
    if generation == GENERATIONS - 1:
        break
    
    # Create next generation
    print(f"  Creating next generation...")
    next_population = population[:ELITE_SIZE]
    
    while len(next_population) < POPULATION_SIZE:
        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)
        child1, child2 = crossover(parent1, parent2)
        child1 = mutate(child1)
        child2 = mutate(child2)
        next_population.extend([child1, child2])
    
    population = next_population[:POPULATION_SIZE]
    print(f"  ✓ Next generation created\n")

print("\n" + "="*80)
print("GENETIC ALGORITHM COMPLETE")
print("="*80)

Initializing population of 20 individuals...
✓ Population initialized

GENERATION 1/15
  [1/20] Evaluating individual...
      Validation RMSE: 0.011577
  [2/20] Evaluating individual...
      Validation RMSE: 0.353441
  [3/20] Evaluating individual...
      Validation RMSE: 0.008257
  [4/20] Evaluating individual...
      Validation RMSE: 0.007868
  [5/20] Evaluating individual...
      Validation RMSE: 0.007467
  [6/20] Evaluating individual...
      Validation RMSE: 0.323709
  [7/20] Evaluating individual...
      Validation RMSE: 0.011544
  [8/20] Evaluating individual...
      Validation RMSE: 0.010916
  [9/20] Evaluating individual...
      Validation RMSE: 0.011278
  [10/20] Evaluating individual...
      Validation RMSE: 0.015419
  [11/20] Evaluating individual...
      Validation RMSE: 0.394954
  [12/20] Evaluating individual...
      Validation RMSE: 0.434980
  [13/20] Evaluating individual...
      Validation RMSE: 0.011290
  [14/20] Evaluating individual...
      Validation

INFO:tensorflow:Assets written to: ram://5f4c1764-4a2d-4d7e-8435-9519fa713d55/assets


INFO:tensorflow:Assets written to: ram://5f4c1764-4a2d-4d7e-8435-9519fa713d55/assets


FileNotFoundError: Unsuccessful TensorSliceReader constructor: Failed to find any matching files for ram://3db9b808-a1ec-4e3f-9611-ff4436559dee/variables/variables
 You may be trying to load on a different device from the computational device. Consider setting the `experimental_io_device` option in `tf.saved_model.LoadOptions` to the io_device such as '/job:localhost'.

## 7. Evaluate Best Model

In [ ]:
best_individual = population[0]

print(f"Best Model Configuration:")
print(f"  CNN Layers: {best_individual.genes['n_cnn_layers']}")
print(f"  CNN Filters: {best_individual.genes['cnn_filters'][:best_individual.genes['n_cnn_layers']]}")
print(f"  Kernel Size: {best_individual.genes['kernel_size']}")
print(f"  CNN Activation: {best_individual.genes['cnn_activation']}")
print(f"  LSTM Layers: {best_individual.genes['n_lstm_layers']}")
print(f"  LSTM Units: {best_individual.genes['lstm_units'][:best_individual.genes['n_lstm_layers']]}")
print(f"  LSTM Activation: {best_individual.genes['lstm_activation']}")
print(f"  Dropout Rate: {best_individual.genes['dropout_rate']:.3f}")
print(f"  Dense Units: {best_individual.genes['dense_units']}")
print(f"  Learning Rate: {best_individual.genes['learning_rate']:.6f}")
print(f"  Batch Size: {best_individual.genes['batch_size']}")

# Test set evaluation
y_pred_test = best_individual.model.predict(X_test_scaled, verbose=0).flatten()
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae = mean_absolute_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)

print(f"\nTest Set Performance:")
print(f"  RMSE: {test_rmse:.6f}")
print(f"  MAE:  {test_mae:.6f}")
print(f"  R²:   {test_r2:.6f}")

# Save
best_individual.model.save('models_ga/best_ga_model.keras')
with open('models_ga/best_genes.json', 'w') as f:
    json.dump(best_individual.genes, f, indent=2)

print(f"\n✓ Best model saved")

## 8. Visualize Evolution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Evolution progress
axes[0, 0].plot(evolution_history['generations'], 
               [-f for f in evolution_history['best_fitness']], 
               'b-o', linewidth=2, markersize=8, label='Best RMSE')
axes[0, 0].plot(evolution_history['generations'], 
               [-f for f in evolution_history['avg_fitness']], 
               'r--s', linewidth=2, markersize=6, label='Avg Fitness')
axes[0, 0].set_xlabel('Generation', fontsize=12)
axes[0, 0].set_ylabel('Validation RMSE', fontsize=12)
axes[0, 0].set_title('Evolution Progress', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Best RMSE per generation
axes[0, 1].bar(evolution_history['generations'], evolution_history['best_rmse'], 
              color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Generation', fontsize=12)
axes[0, 1].set_ylabel('Best Validation RMSE', fontsize=12)
axes[0, 1].set_title('Best Individual per Generation', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Predictions
cycles_test = cycle_ids[test_mask]
axes[1, 0].plot(cycles_test, y_test, 'bo-', linewidth=2, label='Actual', markersize=6)
axes[1, 0].plot(cycles_test, y_pred_test, 'rs--', linewidth=2, label='Predicted', markersize=6)
axes[1, 0].axhline(y=0.7, color='g', linestyle=':', linewidth=2, label='70% Threshold')
axes[1, 0].set_xlabel('Cycle', fontsize=12)
axes[1, 0].set_ylabel('SoH', fontsize=12)
axes[1, 0].set_title(f'Test Predictions (RMSE: {test_rmse:.6f})', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Error distribution
errors = y_pred_test - y_test
axes[1, 1].hist(errors, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Prediction Error', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)
axes[1, 1].set_title('Error Distribution', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('output_ga/ga_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualizations saved")

## 9. Compare with Bayesian Optimization

Load previous Bayesian results and compare

In [ ]:
# Load Bayesian results (if available)
try:
    with open('../output_improved/results.json', 'r') as f:
        bayesian_results = json.load(f)
    
    bayesian_rmse = bayesian_results['sequential']['test']['rmse']
    
    print("Comparison: Genetic Algorithm vs Bayesian Optimization")
    print("="*60)
    print(f"Bayesian Optimization:  Test RMSE = {bayesian_rmse:.6f}")
    print(f"Genetic Algorithm:      Test RMSE = {test_rmse:.6f}")
    print()
    
    if test_rmse < bayesian_rmse:
        improvement = (bayesian_rmse - test_rmse) / bayesian_rmse * 100
        print(f"✓ GA is BETTER by {improvement:.2f}%")
    elif test_rmse > bayesian_rmse:
        degradation = (test_rmse - bayesian_rmse) / bayesian_rmse * 100
        print(f"✗ GA is WORSE by {degradation:.2f}%")
    else:
        print(f"= Same performance")
        
except FileNotFoundError:
    print("Bayesian results not found. Run the previous notebook first.")

## 10. Save Results

In [ ]:
results = {
    'algorithm': 'Genetic Algorithm',
    'configuration': {
        'population_size': POPULATION_SIZE,
        'generations': GENERATIONS,
        'crossover_rate': CROSSOVER_RATE,
        'mutation_rate': MUTATION_RATE,
        'elite_size': ELITE_SIZE
    },
    'best_genes': best_individual.genes,
    'performance': {
        'test_rmse': float(test_rmse),
        'test_mae': float(test_mae),
        'test_r2': float(test_r2),
        'validation_rmse': float(-best_individual.fitness)
    },
    'evolution_history': {
        'generations': evolution_history['generations'],
        'best_rmse': evolution_history['best_rmse'],
        'best_fitness': evolution_history['best_fitness'],
        'avg_fitness': evolution_history['avg_fitness']
    }
}

with open('output_ga/ga_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Results saved to output_ga/ga_results.json")
print("\n" + "="*80)
print("NOTEBOOK COMPLETE")
print("="*80)